# 17 Expanded CAGE-4 SCI Experiment

Run this only when you want the larger paper-grade CAGE-4 evidence budget. It reruns baselines, CybSAGE, SOTA comparison, and the SCI readiness gate with 20 seeds and 3 episodes. Use CPU. This can take time.


In [1]:
from pathlib import Path
import os, sys

REPO_ROOT_OVERRIDE = None


def _looks_like_repo(base: Path) -> bool:
    return (base / "pyproject.toml").exists() and (base / "src" / "cybsage_marl").exists()


def _mount_drive_if_colab():
    if not Path("/content").exists():
        return
    if Path("/content/drive/MyDrive").exists() or Path("/content/drive/My Drive").exists():
        return
    try:
        from google.colab import drive  # type: ignore
        drive.mount("/content/drive")
    except Exception as exc:
        print("Drive auto-mount failed:", exc)


def _find_repo_root():
    _mount_drive_if_colab()
    candidates = []
    if REPO_ROOT_OVERRIDE:
        candidates.append(Path(REPO_ROOT_OVERRIDE).expanduser())
    if os.environ.get("CYBSAGE_REPO_ROOT"):
        candidates.append(Path(os.environ["CYBSAGE_REPO_ROOT"]).expanduser())

    cwd = Path.cwd().resolve()
    candidates.extend([cwd, *cwd.parents])
    candidates.extend([
        Path("/content/drive/MyDrive/projects/implementation/IMPLEMENTATION"),
        Path("/content/drive/My Drive/projects/implementation/IMPLEMENTATION"),
        Path("/content/drive/MyDrive/IMPLEMENTATION"),
        Path("/content/drive/My Drive/IMPLEMENTATION"),
        Path("/content/drive/MyDrive/projects/implementation/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
        Path("/content/drive/My Drive/projects/implementation/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
        Path("/Volumes/LocalDrive1/onlyscholar-projects/CybSage-MARL[DeepakSir]/IMPLEMENTATION"),
    ])

    checked = []
    seen = set()
    for base in candidates:
        base = base.expanduser()
        if str(base) in seen:
            continue
        seen.add(str(base))
        checked.append(base)
        if _looks_like_repo(base):
            return base

    # Last-resort bounded search. This catches Drive shortcuts/copies while avoiding a full Drive crawl.
    search_roots = [Path("/content/drive/MyDrive"), Path("/content/drive/My Drive"), Path("/content")]
    for root in search_roots:
        if not root.exists():
            continue
        try:
            for pyproject in root.glob("**/pyproject.toml"):
                base = pyproject.parent
                checked.append(base)
                if base.name == "IMPLEMENTATION" and _looks_like_repo(base):
                    return base
        except Exception as exc:
            checked.append(Path(f"{root} [search failed: {exc}]"))

    raise RuntimeError(
        "Could not find CybSAGE IMPLEMENTATION repo root visible to this runtime.\n\n"
        "Most common fix: in Colab, run drive.mount('/content/drive') and confirm the folder exists under "
        "/content/drive/MyDrive/projects/implementation/IMPLEMENTATION.\n\n"
        "Checked:\n" + "\n".join(map(str, checked[-80:]))
    )


REPO_ROOT = _find_repo_root()
os.chdir(REPO_ROOT)
src_path = str(REPO_ROOT / "src")
if src_path not in sys.path:
    sys.path.insert(0, src_path)
print("Repository root:", REPO_ROOT)


Mounted at /content/drive
Repository root: /content/drive/MyDrive/projects/implementation/IMPLEMENTATION


In [2]:
SEEDS = list(range(42, 62))
EPISODES = 3
MAX_STEPS = 100
print("Seeds:", SEEDS)
print("Expected CybSAGE eval paired runs:", (len(SEEDS) - len(SEEDS)//2) * EPISODES)


Seeds: [42, 43, 44, 45, 46, 47, 48, 49, 50, 51, 52, 53, 54, 55, 56, 57, 58, 59, 60, 61]
Expected CybSAGE eval paired runs: 30


In [3]:
seed_args = " ".join(map(str, SEEDS))
!PYTHONPATH=src python scripts/run_stage.py cage4_baselines --cage4-episodes {EPISODES} --cage4-max-steps {MAX_STEPS} --cage4-seeds {seed_args}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.13/dist-packages/pkg_resources/__init__.py:3148: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`

In [4]:
seed_args = " ".join(map(str, SEEDS))
!PYTHONPATH=src python scripts/run_stage.py cage4_cybsage --cage4-episodes {EPISODES} --cage4-max-steps {MAX_STEPS} --cage4-seeds {seed_args}


Gym has been unmaintained since 2022 and does not support NumPy 2.0 amongst other critical functionality.
Please upgrade to Gymnasium, the maintained drop-in replacement of Gym, or contact the authors of your software and request that they upgrade.
See the migration guide at https://gymnasium.farama.org/introduction/migration_guide/ for additional information.
/usr/local/lib/python3.13/dist-packages/pygame/pkgdata.py:25: UserWarning: pkg_resources is deprecated as an API. See https://setuptools.pypa.io/en/latest/pkg_resources.html. The pkg_resources package is slated for removal as early as 2025-11-30. Refrain from using this package or pin to Setuptools<81.
  from pkg_resources import resource_stream, resource_exists
/usr/local/lib/python3.13/dist-packages/pkg_resources/__init__.py:3148: DeprecationWarning: Deprecated call to `pkg_resources.declare_namespace('google')`.
Implementing implicit namespace packages (as specified in PEP 420) is preferred to `pkg_resources.declare_namespace`

In [5]:
!PYTHONPATH=src python scripts/run_stage.py cage4_sota


CAGE-4 SOTA comparison summary: {'status': 'completed', 'baselines_compared': 6, 'superiority_claim_ready': True, 'claim_boundary': 'CybSAGE beat every executed CAGE-4 baseline with positive bootstrap confidence intervals.'}


In [6]:
!PYTHONPATH=src python scripts/run_stage.py cage4_sci_readiness --sci-min-paired-runs 30


CAGE-4 SCI readiness summary: {'status': 'completed', 'paper_claim_level': 'SCI-ready superiority claim against executed CAGE-4 baselines', 'superiority_claim_ready': True, 'stage15_superiority_claim_ready': True, 'baselines_compared': 6, 'min_paired_runs': 30, 'recommended_min_paired_runs': 30, 'all_positive_ci': True, 'claim_boundary': 'CybSAGE beat every executed baseline with positive paired bootstrap confidence and enough paired runs.'}


In [7]:
import json, pandas as pd
from pathlib import Path
for folder, filename in [("15_sota_reward_comparison", "cage4_sota_summary.json"), ("16_sci_readiness_pack", "sci_readiness_summary.json")]:
    path = Path("outputs") / folder / filename
    print("\n", path)
    print(json.dumps(json.loads(path.read_text()), indent=2))
print("\nEffect sizes")
pd.read_csv("outputs/16_sci_readiness_pack/cage4_effect_sizes.csv")



 outputs/15_sota_reward_comparison/cage4_sota_summary.json
{
  "status": "completed",
  "baselines_compared": 6,
  "superiority_claim_ready": true,
  "claim_boundary": "CybSAGE beat every executed CAGE-4 baseline with positive bootstrap confidence intervals."
}

 outputs/16_sci_readiness_pack/sci_readiness_summary.json
{
  "status": "completed",
  "paper_claim_level": "SCI-ready superiority claim against executed CAGE-4 baselines",
  "superiority_claim_ready": true,
  "stage15_superiority_claim_ready": true,
  "baselines_compared": 6,
  "min_paired_runs": 30,
  "recommended_min_paired_runs": 30,
  "all_positive_ci": true,
  "claim_boundary": "CybSAGE beat every executed baseline with positive paired bootstrap confidence and enough paired runs."
}

Effect sizes


,baseline,paired_runs,cybsage_mean,baseline_mean,mean_difference,median_difference,ci95_low,ci95_high,win_rate,tie_rate,loss_rate,cohen_d_paired,raw_superiority
0,restore_priority,30,-59.3,-604.566667,545.266667,516.0,514.464167,579.801667,1.000000,0.0,0.000000,5.799647,True
1,random,30,-59.3,-526.333333,467.033333,415.0,407.395000,531.036667,1.000000,0.0,0.000000,2.580341,True
2,remove_priority,30,-59.3,-112.833333,53.533333,40.0,21.800000,88.333333,0.633333,0.0,0.366667,0.566718,True
3,monitor,30,-59.3,-100.300000,41.000000,44.0,17.499167,64.867500,0.633333,0.0,0.366667,0.610229,True
4,sleep,30,-59.3,-100.300000,41.000000,44.0,17.499167,64.867500,0.633333,0.0,0.366667,0.610229,True
5,decoy_priority,30,-59.3,-97.733333,38.433333,36.0,14.566667,62.470000,0.700000,0.0,0.300000,0.578717,True
